In [18]:
import torch
import random 

from util import TokenizerUtil


tokenizer = TokenizerUtil()

input_ids, attention_mask = tokenizer.encode('how are you', max_length=10)

input_ids, attention_mask, tokenizer.decode(input_ids)

(tensor([   0, 9178,   32,   47,    2,    1,    1,    1,    1,    1]),
 tensor([1, 1, 1, 1, 1, 0, 0, 0, 0, 0]),
 '<s>how are you</s>')

In [19]:
from datasets import load_dataset
from transformers import default_data_collator

dataset = load_dataset('json', data_files='dataset/train.json', split='train')
#2,4,4切分,取第0部分
dataset = dataset.select(range(15000))

dataset[0]

{'prompt': 'Human: context= CREATE TABLE table_name_93 (colours VARCHAR, house_name VARCHAR) question= What colours have a House Name of ogun? Assistant:',
 'chosen': 'SELECT colours FROM table_name_93 WHERE house_name = "ogun"',
 'rejected': '',
 'response': 'SELECT colours FROM table_name_93 WHERE house_name = "ogun"'}

In [20]:
def f(data):
    #随机生成两种回答
    if random.random() > 0.5:
        data['chosen'] = data['chosen'].swapcase()
    data = data['prompt'] + data['chosen']

    input_ids, attention_mask = tokenizer.encode(data)

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': input_ids.clone()
    }
    


In [21]:
dataset.column_names

['prompt', 'chosen', 'rejected', 'response']

In [22]:
dataset = dataset.map(f, remove_columns=dataset.column_names)

loader = torch.utils.data.DataLoader(dataset,
                                     collate_fn=default_data_collator,
                                     batch_size=2,
                                     shuffle=True,
                                     drop_last=True)





In [23]:
len(loader), next(iter(loader))

(7500,
 {'input_ids': tensor([[    0, 33837,    35,  ...,     1,     1,     1],
          [    0, 33837,    35,  ...,     1,     1,     1]]),
  'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
          [1, 1, 1,  ..., 0, 0, 0]]),
  'labels': tensor([[    0, 33837,    35,  ...,     1,     1,     1],
          [    0, 33837,    35,  ...,     1,     1,     1]])})

In [ ]:
# 减少参数的量， 因此插入lora
from transformers import AutoModelForCausalLM
import lora
model_actor = AutoModelForCausalLM.from_pretrained(
    'facebook/opt-1.3b',
    use_safetensors=True,  # 加这一行，跳过torch漏洞检查
    dtype="float16"  # 省显存，适配你的6G显卡
)

lora.insert(model_actor)
lora.count_params(model_actor)

`torch_dtype` is deprecated! Use `dtype` instead!


In [ ]:
from transformers import get_scheduler
from accelerate import Accelerator


def f():
    params = []
    params_lora = []
    for name, param in model_actor.named_parameters():
        if not param.requires_grad:
            continue

        if 'lora_A' in name or 'lora_B' in name:
            params_lora.append(param)
            continue

        params.append(param)

    return [{
        'params': params,
        'weight_decay': 0.0,
    }, {
        'params': params_lora,
        'weight_decay': 0.0,
        'lr': 5e-4
    }]


optimizer = torch.optim.Adam(f(), lr=1e-3, betas=(0.9, 0.95))

